# C19 — Ablation du débruitage (D4)

D4 impose de **mesurer** le débruitage avant de l'activer. On compare quatre forces : **sans**,
**0,5**, **0,8** et **1,0** (`prop_decrease` du spectral gating, seuil fixé à 1,5), sur la
**validation**, dans quatre conditions : extrait propre, puis bruité à **20, 10 et 0 dB** de
rapport signal/bruit (bruit mélangé au niveau exact).

Deux mesures, avec intervalles à 95 % par bootstrap (C18) :

- **WER** de faster-whisper (C17) sur la partie `dev` d'OpenSLR 57 (986 phrases) ;
- **F1 macro des émotions** du meilleur modèle (C11 : tête sur Whisper base), sur la partie
  « validation » de CREMA-D + RAVDESS (P1).

**Règle de D4** : on retient la force au meilleur score moyen ; à égalité, la plus faible.

Choix de protocole (à relire par Lynn) :
- le bruit vient de classes d'ESC-50 au fond stable (`configs/denoise_ablation.yaml`) ; un même
  extrait de bruit est utilisé pour toutes les forces : les comparaisons sont appariées ;
- chaîne : extrait (mono 16 kHz, D2) → mélange au SNR → débruitage → modèle. Pas de
  normalisation RMS (D3) ici, comme en C10 et C17 : la condition « sans » + « propre » doit
  retrouver l'ordre de grandeur de C11 et de C17 (contrôle de la section g) ;
- score moyen d'une force = moyenne des deux tâches, chacune moyennée sur les 4 conditions ;
  le score de transcription est 1 − WER (plancher 0), pour que « plus haut = mieux » partout.

## a. Démarrage

On clone le dépôt, on l'installe en mode éditable (I9), puis faster-whisper, CTranslate2 et jiwer (versions de C17), et cuBLAS/cuDNN pour CUDA 12. GPU T4, Internet activé.

In [ ]:
import sys

BRANCH = "exp/c19-denoising-ablation"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
%pip install -q "faster-whisper==1.2.1" "ctranslate2==4.8.5" "jiwer==4.0.0"
%pip install -q "nvidia-cublas-cu12==12.9.3.13" "nvidia-cudnn-cu12==9.28.1.4"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg, GPU et bibliothèques CUDA 12

Même préparation qu'en C17 (f) : cuBLAS et cuDNN chargés par leur chemin avant le premier calcul de CTranslate2.

In [ ]:
import ctypes
import importlib.util
import shutil
import time
from pathlib import Path

import ctranslate2
import torch

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
NOTEBOOK_START = time.time()


def nvidia_library_dir(package):
    spec = importlib.util.find_spec(package)
    return Path(spec.submodule_search_locations[0]) / "lib"


def preload_libraries(directory):
    # Chargement global, en plusieurs passes : une bibliothèque peut dépendre d'une autre.
    pending = sorted(directory.glob("lib*.so*"))
    for _ in range(len(pending)):
        failed = []
        for path in pending:
            try:
                ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                failed.append(path)
        if len(failed) == len(pending):
            break
        pending = failed
    return [path.name for path in pending]


if ctranslate2.get_cuda_device_count() == 0 or not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4.")
for package in ["nvidia.cublas", "nvidia.cudnn"]:
    print(package, "non chargées :", preload_libraries(nvidia_library_dir(package)))
DEVICE = torch.device("cuda")
print("GPU :", torch.cuda.get_device_name(0))

## c. Jeux de données

CREMA-D et RAVDESS (datasets Kaggle attachés), ESC-50 (bruits ; archive officielle) et OpenSLR 57
(archive officielle, ~1,8 Go, décompressée dans `/tmp`). Liens dans `/kaggle/working/datasets/<id>`
(contrat des manifestes, `docs/datasets.md`).

In [ ]:
import tarfile
import urllib.request
import zipfile

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
DOWNLOAD_DIR = Path("/kaggle/working/downloads")
OPENSLR_DIR = Path("/tmp/openslr57")
ESC50_ARCHIVE_URL = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
OPENSLR57_TAR_URL = "https://openslr.elda.org/resources/57/African_Accented_French.tar.gz"
CREMAD_EXPECTED_WAV = 7442
RAVDESS_EXPECTED_WAV = 1440


def find_cremad_root():
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        if len(list(audio_folder.glob("*.wav"))) == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    raise RuntimeError("CREMA-D introuvable : attacher le dataset ejlok1/cremad.")


def find_ravdess_root():
    for candidate in sorted({actor.parent for actor in INPUT_DIR.glob("**/Actor_01")}):
        wav_paths = list(candidate.glob("Actor_*/*.wav"))
        if len(wav_paths) == RAVDESS_EXPECTED_WAV:
            return candidate
    raise RuntimeError("RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.")


def download_esc50():
    found = sorted(INPUT_DIR.glob("**/meta/esc50.csv"))
    if found:
        return found[0].parent.parent
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = DOWNLOAD_DIR / "esc50.zip"
    urllib.request.urlretrieve(ESC50_ARCHIVE_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DOWNLOAD_DIR)
    archive_path.unlink()
    return DOWNLOAD_DIR / "ESC-50-master"


def download_openslr57():
    OPENSLR_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = OPENSLR_DIR / "openslr57.tar.gz"
    urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(OPENSLR_DIR, filter="data")
    archive_path.unlink()
    return sorted(OPENSLR_DIR.glob("**/African_Accented_French"))[0]


roots = {
    "esc50": download_esc50(),
    "cremad": find_cremad_root(),
    "ravdess": find_ravdess_root(),
    "openslr57": download_openslr57(),
}
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
for dataset_id, root in roots.items():
    link = DATASETS_DIR / dataset_id
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(dataset_id, "→", root)

## d. Configuration et jeux de validation

Les paramètres sont dans `configs/denoise_ablation.yaml` (aucun nombre dans le code). On lit :
la validation des émotions (découpage P1 versionné, classes communes P2) et la partie `dev`
d'OpenSLR 57. On vérifie que chaque audio existe.

In [ ]:
import numpy as np

from echolens.config import CONFIG_DIR, REPO_ROOT
from echolens.core.audio import load_audio
from echolens.core.audio.preprocess_config import load_preprocess_config
from echolens.core.data.emotions import load_emotion_labels
from echolens.core.data.manifest import audio_path, load_manifest
from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config
from echolens.training.denoise_ablation import load_ablation_config
from echolens.training.evaluation import EvaluationConfig
from echolens.training.heads import emotion_examples, load_heads_config
from echolens.training.transcription_eval import (
    CorpusSelection,
    corpus_rows,
    load_transcription_eval_config,
    sample_rows,
)

ablation = load_ablation_config()
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
transcription_config = load_transcription_eval_config()
base_denoise_config = load_preprocess_config().denoise
labels = load_emotion_labels()
split_config = load_split_config()
speaker_splits = load_speaker_splits(REPO_ROOT / "data" / "splits" / "emotion_split.csv")
check_no_speaker_leak(speaker_splits)
manifests_dir = REPO_ROOT / "data" / "manifests"

# Étape 1 : extraits d'émotions de la validation, dans l'ordre des manifestes.
emotion_clips = []
for dataset_id in load_heads_config().emotion_datasets:
    rows = load_manifest(manifests_dir / f"{dataset_id}.csv")
    for example in emotion_examples(rows, labels, speaker_splits, split_config):
        if example.split == "validation":
            emotion_clips.append((rows[example.position], example.target))
print("Émotions, validation :", len(emotion_clips), "extraits")

# Étape 2 : phrases de la partie dev d'OpenSLR 57.
openslr_rows = load_manifest(manifests_dir / "openslr57.csv")
dev_rows = corpus_rows(openslr_rows, CorpusSelection(id="dev", official_split="dev"))
if ablation.wer_sentences is not None:
    dev_rows = sample_rows(dev_rows, ablation.wer_sentences, ablation.seed)
print(
    "Transcription, dev :",
    len(dev_rows),
    "phrases,",
    len({row.speaker for row in dev_rows}),
    "locuteurs",
)

# Étape 3 : tous les audios sont là.
all_rows = [row for row, _ in emotion_clips] + dev_rows
missing = [row.path for row in all_rows if not audio_path(row, DATASETS_DIR).exists()]
if missing:
    raise RuntimeError(f"{len(missing)} audios absents, par exemple {missing[:3]}")

## e. Bruits, SNR exact et chaîne de traitement

Les extraits de bruit sont les classes `noise_classes` d'ESC-50 (rain, wind, vacuum_cleaner, …),
lus avec `load_audio`. Pour l'extrait numéro *i*, le générateur de graine `seed + i` tire un bruit
et un point de départ : **le bruit est le même pour toutes les forces**. Le gain du bruit est
calculé pour que le SNR soit exact (`mix_at_snr`, testé à la main dans
`tests/unit/test_denoise_ablation.py`). Vérification : le SNR mesuré doit valoir 20, 10 et 0 dB.

In [ ]:
from echolens.core.audio.denoise import denoise
from echolens.training.denoise_ablation import (
    fit_noise_length,
    measured_snr_db,
    mix_at_snr,
    pick_noise_clip,
)

# Étape 1 : le lot de bruits d'ESC-50.
esc50_rows = load_manifest(manifests_dir / "esc50.csv")
noise_pool = []
for row in esc50_rows:
    if row.label in ablation.noise_classes:
        noise_pool.append(load_audio(audio_path(row, DATASETS_DIR)))
print("Bruits :", len(noise_pool), "extraits de", ablation.noise_classes)


def noise_for_clip(clip_index, length):
    # Étape 2 : bruit de la bonne longueur, toujours le même pour un extrait donné.
    rng = np.random.default_rng(ablation.seed + clip_index)
    noise = pick_noise_clip(noise_pool, ablation.noise_min_dbfs, ablation.noise_max_tries, rng)
    start = int(rng.integers(0, len(noise)))
    return fit_noise_length(noise, length, start)


def denoise_config_for(strength):
    # Force None = « sans » : enabled False, le signal est renvoyé inchangé (denoise.py).
    if strength is None:
        return base_denoise_config.model_copy(update={"enabled": False})
    return base_denoise_config.model_copy(update={"enabled": True, "prop_decrease": strength})


def prepare(signal, clip_index, snr_db, strength):
    # Étape 3 : mélange au SNR (sauf extrait propre), puis débruitage.
    if snr_db is not None:
        noise = noise_for_clip(clip_index, len(signal))
        signal = mix_at_snr(signal, noise, snr_db)
    return denoise(signal, denoise_config_for(strength))


# Contrôle du SNR sur un extrait de parole.
probe_signal = load_audio(audio_path(dev_rows[0], DATASETS_DIR))
for snr in [20, 10, 0]:
    mixed = prepare(probe_signal, 0, snr, None)
    print(f"SNR demandé {snr:2d} dB → mesuré {measured_snr_db(probe_signal, mixed):.3f} dB")

## f. Mesures

Une **cellule** = une force × une condition (16 au total). Les résultats de chaque cellule sont
écrits dans `c19_cells.json` dès qu'elle est finie : si Kaggle coupe la session, une relance (avec
cette sortie ajoutée en entrée du notebook) reprend où le calcul s'était arrêté.

- Émotions : encodeur Whisper base gelé (sur GPU) → moyenne des couches sur les trames utiles →
  standardisation de l'entraînement (R4) → tête du registre (poids à la révision figée) → classe.
  Même code que `ModelPredictor`, mais l'encodeur tourne sur GPU.
- Transcription : faster-whisper float16, réglages de `configs/transcription.yaml`, normalisation
  P12, WER avec intervalle par bootstrap **par locuteur** (C17).

In [ ]:
import json

from huggingface_hub import hf_hub_download

from echolens.core.models.heads import LayerWeightedHead
from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config
from echolens.core.models.registry import ModelRegistry
from echolens.core.models.storage import load_weights
from echolens.core.models.whisper_encoder import load_whisper_encoder, whisper_useful_layer_means

# Étape 1 : tête des émotions, à la révision figée du registre (configs/registry.yaml).
registry = ModelRegistry.from_yaml(CONFIG_DIR / "registry.yaml")
matching_entries = [model for model in registry.models if model.id == ablation.emotion_model_id]
entry = matching_entries[0]
repo_id = entry.weights.removeprefix("hf://")
model_config = load_production_config(
    Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision))
)
weights_path = Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision))
if model_config.labels != [label.id for label in labels.emotions]:
    raise RuntimeError("Classes du modèle différentes de configs/labels.yaml")
shape = model_config.head
head = LayerWeightedHead(
    shape.layer_count, shape.dimension, shape.hidden_size, len(model_config.labels), shape.dropout
)
head = load_weights(head, weights_path)
head_mean = np.asarray(model_config.standardization_mean, dtype=np.float32)
head_std = np.asarray(model_config.standardization_std, dtype=np.float32)

# Étape 2 : encodeur Whisper gelé, sur GPU.
reference = model_config.encoder
encoder = load_whisper_encoder(
    reference.checkpoint,
    reference.revision,
    reference.sample_rate,
    reference.samples_per_output_frame,
    reference.max_output_frames,
    DEVICE,
)
print(entry.id, "· révision", entry.revision[:8], "· F1 test du registre", entry.metrics)


def predict_emotion(signal):
    means = whisper_useful_layer_means(signal, encoder)
    standardized = ((means - head_mean) / head_std).astype(np.float32)
    inputs = torch.from_numpy(standardized).unsqueeze(0)
    with torch.no_grad():
        scores = head(inputs)
    return int(scores[0].argmax())

In [ ]:
from echolens.training.transcription_eval import score_sentence
from echolens.training.transcription_models import load_whisper, transcribe

matching_models = [m for m in transcription_config.models if m.id == ablation.whisper_model_id]
whisper_model_config = matching_models[0]
whisper = load_whisper(whisper_model_config, "cuda", "float16", transcription_config)
print(
    "Whisper",
    whisper_model_config.id,
    "chargé sur GPU, révision",
    whisper_model_config.revision[:8],
)

In [ ]:
OUTPUT_DIR = Path("/kaggle/working")
CELLS_PATH = OUTPUT_DIR / "c19_cells.json"


def cell_key(strength, snr_db):
    strength_name = "none" if strength is None else str(strength)
    snr_name = "clean" if snr_db is None else str(snr_db)
    return f"{strength_name}|{snr_name}"


# Reprise : cellules déjà calculées (sortie d'un passage précédent, ajoutée en entrée).
cells = {}
for previous in sorted(INPUT_DIR.glob("**/c19_cells.json")):
    if not CELLS_PATH.exists():
        shutil.copy(previous, CELLS_PATH)
if CELLS_PATH.exists():
    cells = json.loads(CELLS_PATH.read_text(encoding="utf-8"))
print(len(cells), "cellules déjà calculées sur", len(ablation.strengths) * len(ablation.snr_db))


def run_emotion_cell(strength, snr_db):
    predictions = []
    for clip_index, (row, _) in enumerate(emotion_clips):
        signal = load_audio(audio_path(row, DATASETS_DIR))
        prepared = prepare(signal, clip_index, snr_db, strength)
        predictions.append(predict_emotion(prepared))
    return predictions


def run_wer_cell(strength, snr_db):
    hypotheses = []
    for clip_index, row in enumerate(dev_rows):
        signal = load_audio(audio_path(row, DATASETS_DIR))
        # Indices décalés : le bruit d'une phrase ne reprend pas celui d'un extrait d'émotion.
        prepared = prepare(signal, 100_000 + clip_index, snr_db, strength)
        hypotheses.append(transcribe(whisper, prepared, transcription_config))
    return hypotheses


for strength in ablation.strengths:
    for snr_db in ablation.snr_db:
        key = cell_key(strength, snr_db)
        if key in cells:
            continue
        start = time.perf_counter()
        emotion_predictions = run_emotion_cell(strength, snr_db)
        hypotheses = run_wer_cell(strength, snr_db)
        cells[key] = {"emotion_predictions": emotion_predictions, "hypotheses": hypotheses}
        CELLS_PATH.write_text(json.dumps(cells, ensure_ascii=False), encoding="utf-8")
        print(f"{key:12s} fait en {time.perf_counter() - start:.0f} s")

## g. Scores, intervalles et contrôle

Pour chaque cellule : F1 macro des émotions avec intervalle (`bootstrap_ci`, C18), WER avec
intervalle par locuteur. **Contrôle** : la cellule « sans / propre » doit être proche du F1 de
validation de C11 (`chosen.validation_f1` de `training/results/c11_family_b_emotions.json`,
≈ 0,745 ; la tête du registre vient de C13 et peut différer un peu) et d'un WER de même ordre
que ceux de C17 sur `test` ; un grand écart signale une chaîne différente de celle de
l'entraînement (à corriger avant de lire le reste).

In [ ]:
import pandas as pd

from echolens.training.evaluation import bootstrap_ci, macro_f1
from echolens.training.transcription_eval import bootstrap_intervals

emotion_targets = np.array([target for _, target in emotion_clips], dtype=np.int_)
results = {}
for strength in ablation.strengths:
    for snr_db in ablation.snr_db:
        key = cell_key(strength, snr_db)
        cell = cells[key]
        predictions = np.array(cell["emotion_predictions"], dtype=np.int_)
        f1 = macro_f1(emotion_targets, predictions)
        f1_low, f1_high = bootstrap_ci(emotion_targets, predictions, evaluation_config, macro_f1)
        scores = []
        for row, hypothesis in zip(dev_rows, cell["hypotheses"], strict=True):
            scores.append(score_sentence(row.path, row.speaker, row.transcript, hypothesis))
        wer, _ = bootstrap_intervals(
            scores,
            transcription_config.bootstrap_draws,
            transcription_config.seed,
            transcription_config.confidence_level,
        )
        results[key] = {
            "emotion_f1": [f1, f1_low, f1_high],
            "wer": [wer.value, wer.low, wer.high],
        }

rows_for_table = []
for key, value in results.items():
    strength_name, snr_name = key.split("|")
    rows_for_table.append(
        {
            "force": strength_name,
            "condition": snr_name if snr_name == "clean" else f"{snr_name} dB",
            "F1 émotions": (
                f"{value['emotion_f1'][0]:.3f} "
                f"[{value['emotion_f1'][1]:.3f} ; {value['emotion_f1'][2]:.3f}]"
            ),
            "WER": f"{value['wer'][0]:.1%} [{value['wer'][1]:.1%} ; {value['wer'][2]:.1%}]",
        }
    )
table = pd.DataFrame(rows_for_table)
display(table)
table.to_csv(OUTPUT_DIR / "c19_table.csv", index=False)

reference_key = cell_key(None, None)
print(
    "Contrôle « sans / propre » → F1 émotions",
    round(results[reference_key]["emotion_f1"][0], 4),
    "· WER",
    round(results[reference_key]["wer"][0], 4),
)
c11 = json.loads(
    (REPO_ROOT / "training" / "results" / "c11_family_b_emotions.json").read_text(encoding="utf-8")
)
print("C11, F1 de validation de la tête retenue :", c11["models"]["whisper-base"]["chosen"])

## h. Choix de la force (règle de D4)

Score d'une force = moyenne, sur les quatre conditions, du F1 des émotions et de 1 − WER
(plancher 0), puis moyenne des deux tâches. On affiche aussi la décision tâche par tâche : si
elles divergent, c'est à noter dans le rapport (le débruitage peut aider la parole et abîmer les
émotions, hypothèse de D4).

In [ ]:
from echolens.training.denoise_ablation import choose_strength


def strength_name(strength):
    return "none" if strength is None else str(strength)


emotion_mean = {}
transcription_mean = {}
for strength in ablation.strengths:
    f1_values = []
    wer_scores = []
    for snr_db in ablation.snr_db:
        value = results[cell_key(strength, snr_db)]
        f1_values.append(value["emotion_f1"][0])
        wer_scores.append(max(0.0, 1.0 - value["wer"][0]))
    emotion_mean[strength_name(strength)] = float(np.mean(f1_values))
    transcription_mean[strength_name(strength)] = float(np.mean(wer_scores))

overall = {}
for name in emotion_mean:
    overall[name] = (emotion_mean[name] + transcription_mean[name]) / 2

chosen, rounded = choose_strength(overall, ablation.tie_decimals)
chosen_emotion, _ = choose_strength(emotion_mean, ablation.tie_decimals)
chosen_transcription, _ = choose_strength(transcription_mean, ablation.tie_decimals)
summary = pd.DataFrame(
    {
        "F1 émotions (moy.)": emotion_mean,
        "1 − WER (moy.)": transcription_mean,
        "score moyen": overall,
    }
).round(4)
display(summary)
print("Force retenue (règle de D4) :", chosen)
print("  si on ne regardait que les émotions :", chosen_emotion)
print("  si on ne regardait que la transcription :", chosen_transcription)

## i. Figure pour le rapport

Score selon le niveau de bruit, une courbe par force : WER à gauche (plus bas = mieux), F1 des émotions à droite, barres = intervalles à 95 %.

In [ ]:
import matplotlib.pyplot as plt

condition_names = ["propre" if snr is None else f"{snr} dB" for snr in ablation.snr_db]
positions = np.arange(len(condition_names))
figure, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
panels = [("wer", "WER (↓)", axes[0]), ("emotion_f1", "F1 macro des émotions (↑)", axes[1])]
for metric, title, axis in panels:
    for index, strength in enumerate(ablation.strengths):
        values = []
        lows = []
        highs = []
        for snr_db in ablation.snr_db:
            value, low, high = results[cell_key(strength, snr_db)][metric]
            values.append(value)
            lows.append(value - low)
            highs.append(high - value)
        shift = (index - 1.5) * 0.06
        label = "sans" if strength is None else f"force {strength}"
        axis.errorbar(
            positions + shift, values, yerr=[lows, highs], marker="o", capsize=3, label=label
        )
    axis.set_title(title)
    axis.set_xticks(positions, condition_names)
    axis.set_xlabel("niveau de bruit (SNR)")
    axis.grid(alpha=0.3)
axes[0].legend()
figure.tight_layout()
FIGURE_PATH = OUTPUT_DIR / "c19_denoising_ablation.png"
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()

## j. Résultats et paragraphe pour le rapport

`c19_denoising_ablation.json` (→ `training/results/`), `c19_denoising_ablation.png`
(→ `report/figures/`). Le choix s'applique ensuite dans `configs/preprocess.yaml` :
`enabled: true` et `prop_decrease: <force>` si une force gagne, `enabled: false` si « sans » gagne
(avec le commentaire de la règle de D4 et ce résultat).

In [ ]:
import datetime
import platform
import subprocess

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
content = {
    "experiment": "c19_denoising_ablation",
    "date": datetime.datetime.now(datetime.UTC).isoformat(),
    "branch": BRANCH,
    "commit": commit,
    "python": platform.python_version(),
    "gpu": torch.cuda.get_device_name(0),
    "config": ablation.model_dump(),
    "denoise_fixed": {
        "stationary": base_denoise_config.stationary,
        "n_std_thresh_stationary": base_denoise_config.n_std_thresh_stationary,
        "n_fft": base_denoise_config.n_fft,
    },
    "emotion_model": {
        "id": entry.id,
        "revision": entry.revision,
        "validation_size": len(emotion_clips),
    },
    "whisper": {
        "id": whisper_model_config.id,
        "revision": whisper_model_config.revision,
        "sentences": len(dev_rows),
    },
    "cells": results,
    "mean_by_strength": {
        "emotion_f1": emotion_mean,
        "one_minus_wer": transcription_mean,
        "overall": overall,
    },
    "chosen_strength": chosen,
    "chosen_by_task": {"emotions": chosen_emotion, "transcription": chosen_transcription},
    "runtime_s": round(time.time() - NOTEBOOK_START),
}
results_path = OUTPUT_DIR / "c19_denoising_ablation.json"
results_path.write_text(json.dumps(content, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(results_path)

best = results[cell_key(None, 0)]
chosen_value = None if chosen == "none" else float(chosen)
chosen_zero_db = results[cell_key(chosen_value, 0)]
print()
print("Brouillon de paragraphe (à relire et corriger) :")
print(
    f"L'ablation du débruitage (D4) compare quatre forces (sans, 0,5, 0,8, 1,0) sur la validation, "
    f"propre et bruitée à 20, 10 et 0 dB. À 0 dB, sans débruitage : WER {best['wer'][0]:.1%}, "
    f"F1 des émotions {best['emotion_f1'][0]:.3f} ; avec la force retenue ({chosen}) : "
    f"WER {chosen_zero_db['wer'][0]:.1%}, F1 {chosen_zero_db['emotion_f1'][0]:.3f}. "
    "La règle de D4 (meilleur score moyen, la plus faible force en cas d'égalité) "
    f"retient : {chosen}."
)